## PGA – GradCAM Rescue + IPR

**Yêu cầu**: Đã train model PGA-UNet và lưu checkpoint lên Google Drive.

Chạy tuần tự 3 cells: Setup → Download checkpoint → Rescue Pipeline

In [ ]:
# =========================================================
# CELL 1 - SETUP: Clone repo + Download dataset
# =========================================================
%cd /content

# Clone repo (branch TN_B_ON)
!git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# Download dataset từ Google Drive
!gdown --id 1fU7KPln7joaa3EZZtGn-VKeg9i4AmPG3

# Giải nén + di chuyển vào project
!unzip -oq dataset_BTXRD.zip
!mv dataset_BTXRD Prompt-Guided-XRay-Segmentation/

# Vào thư mục project
%cd Prompt-Guided-XRay-Segmentation

# Cài dependencies
!pip install -q tqdm opencv-python matplotlib scikit-image gdown scipy

print("\n✅ Setup done!")


## Bước 1 – Tải checkpoint từ Google Drive

In [ ]:
import gdown, os

# =========================================================
# ĐIỀN FILE_ID GOOGLE DRIVE CỦA FILE .pth VÀO ĐÂY
# Cách lấy: mở file trên Drive → Share → Copy link
#   Link:    https://drive.google.com/file/d/FILE_ID/view
#   File_id: phần giữa /d/ và /view
# =========================================================
CKPT_FILE_ID = "YOUR_FILE_ID_HERE"   # ← SỬA DÒNG NÀY

SAVE_PATH = "checkpoints/pga_unet_expB_best.pth"
os.makedirs("checkpoints", exist_ok=True)

assert CKPT_FILE_ID != "YOUR_FILE_ID_HERE",     "❌ Hãy điền file_id Google Drive của checkpoint .pth trước khi chạy!"

url = f"https://drive.google.com/uc?id={CKPT_FILE_ID}"
gdown.download(url, SAVE_PATH, quiet=False)

import os.path
assert os.path.exists(SAVE_PATH), f"❌ Tải thất bại: {SAVE_PATH}"
print(f"✅ Checkpoint sẵn sàng: {SAVE_PATH}  ({os.path.getsize(SAVE_PATH)//1024} KB)")


## Bước 2 – GradCAM Rescue + IPR Pipeline

Kiểm tra 3 điều kiện bảo hộ, tính Achievement 1, phân 4 nhóm, đánh giá 6 độ đo.

In [ ]:
# @title  Thí Nghiệm B – GradCAM Rescue + IPR (174 mẫu nền đen)
import os, cv2, torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

# ═══════════════════════════════════════════════════════════════════════
# 1. CẤU HÌNH
# ═══════════════════════════════════════════════════════════════════════
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH           = "checkpoints/pga_unet_expB_best.pth"
IMG_SIZE             = 512
USE_ENCODER_PROMPT   = True
NUM_IPR_STEPS        = 3      # steps[0]=GradCAM pred, steps[1]=IPR V1, steps[2]=IPR V2
NUM_VIS_PER_BIN      = 3
# ── Điều kiện kích hoạt bảo hộ (tất cả 3 phải thỏa) ─────────────────
DARK_RATIO_LIMIT     = 0.70   # ①  Prompt >70% đen (góc ảnh)
DARK_PIXEL_THRESHOLD = -0.80  #    Ngưỡng pixel "tối" sau chuẩn hóa [-1,1]
PRED_PIXEL_LIMIT     = 50     # ②  Prediction <50px
CONF_LIMIT           = 0.25   # ③  Confidence (max sigmoid) <0.25
TEST_IMAGE_DIR       = "dataset_BTXRD/test/images"
TEST_JSON_DIR        = "dataset_BTXRD/test/annotations"
BINS = [
    ("0.0 – 0.5",  0.0, 0.50, "tomato"),
    ("0.5 – 0.7",  0.5, 0.70, "gold"),
    ("0.7 – 0.9",  0.7, 0.90, "deepskyblue"),
    ("0.9 – 0.99", 0.9, 1.00, "limegreen"),  # [0.9,1.0) loại trừ Dice=1.0
]

# ═══════════════════════════════════════════════════════════════════════
# 2. HELPERS
# ═══════════════════════════════════════════════════════════════════════
def extract_lcc(binary_map: np.ndarray) -> np.ndarray:
    if binary_map.sum() == 0: return binary_map
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary_map.astype(np.uint8), 8)
    if n <= 1: return binary_map
    return (labels == (1 + np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_hd95(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred); ge = gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_all_metrics(prob: np.ndarray, gm: np.ndarray) -> dict:
    pm  = extract_lcc((prob > 0.5).astype(np.float32))
    gm2 = (gm  > 0.5).astype(np.float32)
    eps = 1e-6
    tp  = (pm*gm2).sum(); fp = (pm*(1-gm2)).sum(); fn = ((1-pm)*gm2).sum()
    dice = (2*tp+eps)/(2*tp+fp+fn+eps); iou  = (tp+eps)/(tp+fp+fn+eps)
    prec = (tp+eps)/(tp+fp+eps);        rec  = (tp+eps)/(tp+fn+eps)
    hd95 = calc_hd95(pm, gm2)
    if gm2.sum()==0 or pm.sum()==0:
        cbl = 0.0
    else:
        ys,xs = np.where(gm2>0.5); yp,xp = np.where(pm>0.5)
        gt_diag = np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl = float(np.clip(1.0 - np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag, 0.0, 1.0))
    return dict(mask=pm, dice=float(dice), iou=float(iou),
                precision=float(prec), recall=float(rec), hd95=hd95, cbl=cbl)

def compute_gradcam(model, img_t: torch.Tensor):
    grads, acts = [], []
    h = model.center.register_forward_hook(
        lambda m,i,o: (acts.append(o), o.register_hook(lambda g: grads.append(g))))
    model.eval()
    zero_p = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = model(img_t.clone().detach(), zero_p)
    model.zero_grad(); out.sum().backward(); h.remove()
    if not grads: return None
    w   = grads[0].mean(dim=(2,3), keepdim=True)
    cam = F.relu((w * acts[0]).sum(1, keepdim=True))
    cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
    cam = cam[0,0].detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def get_centroid(pm: np.ndarray):
    if pm.sum() == 0: return None, None
    ys, xs = np.where(pm > 0.5)
    return float(xs.mean()), float(ys.mean())

def is_dark_bg(img_np: np.ndarray) -> bool:
    bw, bh = 80, 80
    corners = [(10,10),(IMG_SIZE-bw-10,10),(10,IMG_SIZE-bh-10),(IMG_SIZE-bw-10,IMG_SIZE-bh-10)]
    best = max((img_np[by:by+bh, bx:bx+bw] < DARK_PIXEL_THRESHOLD).mean() for bx,by in corners)
    return best >= DARK_RATIO_LIMIT

# ═══════════════════════════════════════════════════════════════════════
# 3. VÒNG QUÉT CHÍNH
# ═══════════════════════════════════════════════════════════════════════
def run_full_evaluation(model, test_ds):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    r      = (test_ds.zoom_ratio[0] + test_ds.zoom_ratio[1]) / 2
    model.eval()

    n_cond1       = 0   # thỏa ①
    n_all_conds   = 0   # thỏa ①+②+③
    n_protection  = 0   # trong ①+②+③: zero_prompt → mask rỗng
    per_sample    = []

    for img_t, mask_t, prompt_t in tqdm(loader, desc="🔍 Quét test set"):
        img_np = img_t[0,0].numpy()
        gt_np  = mask_t[0,0].numpy()
        if gt_np.sum() == 0: continue
        img_dev = img_t.to(DEVICE)

        # ── Điều kiện ①: Prompt >70% đen ─────────────────────────────
        if not is_dark_bg(img_np): continue
        n_cond1 += 1

        # ── Chạy model với REAL PROMPT → kiểm tra ②③ ─────────────────
        with torch.no_grad():
            prob_real = torch.sigmoid(model(img_dev, prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        cond2 = int((prob_real > 0.5).sum()) < PRED_PIXEL_LIMIT     # ② Prediction <50px
        cond3 = float(prob_real.max()) < CONF_LIMIT                  # ③ Confidence <0.25
        if not (cond2 and cond3): continue
        n_all_conds += 1

        # ── Bước 1 – Achievement 1: Zero Prompt → xác nhận mask rỗng ──
        with torch.no_grad():
            zero_p    = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
            prob_zero = torch.sigmoid(model(img_dev, zero_p))[0,0].cpu().numpy()
        if (prob_zero > 0.5).sum() == 0: n_protection += 1

        # ── Bước 2 – GradCAM → step 0 (GradCAM prediction) ───────────
        sal = compute_gradcam(model, img_dev)
        if sal is None: continue
        py_c, px_c = np.unravel_index(sal.argmax(), sal.shape)
        ys_g, xs_g = np.where(gt_np > 0.5)
        bw = (xs_g.max()-xs_g.min()) * (1+r)
        bh = (ys_g.max()-ys_g.min()) * (1+r)

        steps = []
        for v in range(NUM_IPR_STEPS):
            pm_map = test_ds.create_plateau_heatmap(
                [px_c-bw/2, py_c-bh/2, px_c+bw/2, py_c+bh/2], IMG_SIZE, IMG_SIZE)
            pm_t = (torch.from_numpy(cv2.resize(pm_map,(IMG_SIZE,IMG_SIZE)))
                       .unsqueeze(0).unsqueeze(0).to(DEVICE))
            with torch.no_grad():
                prob = torch.sigmoid(model(img_dev, pm_t))[0,0].cpu().numpy()
            m = calc_all_metrics(prob, gt_np)
            cx, cy = get_centroid(m["mask"])
            if cx is not None: px_c, py_c = cx, cy
            steps.append({**m, "pm": pm_map, "cx": cx, "cy": cy})

        per_sample.append({
            "img": img_np, "gt": gt_np, "sal": sal,
            "prob0": prob_zero,          # zero-prompt → rỗng (Achievement 1)
            "gcam_dice": steps[0]["dice"] if steps else 0.0,  # để phân nhóm
            "steps": steps, "final": steps[-1] if steps else {}
        })

    # ── In Bước 1 ─────────────────────────────────────────────────────
    rate = n_protection / n_all_conds * 100 if n_all_conds > 0 else 0
    bar  = "═" * 65
    print(f"\n{bar}")
    print("  BƯỚC 1 — XÁC NHẬN BẢO HỘ  (ACHIEVEMENT 1)")
    print(bar)
    print(f"  Mẫu thỏa ①  (Prompt >70% đen)              : {n_cond1}")
    print(f"  Mẫu thỏa ①+②+③ (cần cứu hộ)               : {n_all_conds}")
    print(f"    ② Prediction <{PRED_PIXEL_LIMIT}px  ③ Confidence <{CONF_LIMIT}")
    print(f"  Bảo hộ = Zero Prompt → Mask rỗng            : {n_protection}/{n_all_conds} = {rate:.1f}%")
    tok = "✅" if rate >= 99.0 else "⚠️ "
    print(f"  {tok} {'BẢO HỘ ĐẠT ~100%' if rate>=99.0 else f'Cần kiểm tra thêm {n_all_conds-n_protection} mẫu'}")
    print(f"{bar}\n  Đã chạy GradCAM+IPR: {len(per_sample)} mẫu\n")
    return per_sample

# ═══════════════════════════════════════════════════════════════════════
# 4. BẢNG 6 ĐỘ ĐO THEO NHÓM (GradCAM vs IPR cuối)
# ═══════════════════════════════════════════════════════════════════════
def _bin_groups(per_sample, key="gcam_dice"):
    """Phân nhóm samples dựa vào `key` value."""
    groups = {b[0]: [] for b in BINS}
    for s in per_sample:
        d = s.get(key, s["steps"][0]["dice"]) if key == "gcam_dice" else s["final"].get("dice", 0.0)
        for bname, lo, hi, _ in BINS:
            if lo <= d < hi:
                groups[bname].append(s); break
    return groups

def _metrics_row(samples, step_key):
    """Tính mean 6 metrics từ step_key ('steps[0]' hoặc 'final')."""
    KEYS = ["dice","iou","precision","recall","hd95","cbl"]
    if not samples: return {k: 0.0 for k in KEYS}
    def get_m(s):
        return s["steps"][0] if step_key == "gcam" else s["final"]
    return {k: np.mean([get_m(s)[k] for s in samples if get_m(s)]) for k in KEYS}

def print_per_bin_tables(per_sample):
    KEYS = ["dice","iou","precision","recall","hd95","cbl"]
    HDRS = ["Dice↑","IoU↑","Prec↑","Rec↑","HD95↓","CBL↑"]
    groups = _bin_groups(per_sample, key="gcam_dice")

    for stage, step_key, label in [
        ("GradCAM (bước đầu)", "gcam", "BẢNG 1 — GRADCAM PREDICTION"),
        ("IPR cuối (k=3)",     "final","BẢNG 2 — SAU IPR (k=3)    "),
    ]:
        bar = "═" * 75
        print(f"\n{bar}")
        print(f"  {label}  |  6 ĐỘ ĐO THEO NHÓM")
        print(bar)
        print(f"  {'Nhóm':<13}{'N':>4}  " + "".join(f"{h:>9}" for h in HDRS))
        print(f"  {'─'*71}")
        totals = {k: [] for k in KEYS}
        for bname, lo, hi, _ in BINS:
            g = groups[bname]
            m = _metrics_row(g, step_key)
            row = f"  {bname:<13}{len(g):>4}  " + "".join(
                f"{m[k]:>9.2f}" if k=="hd95" else f"{m[k]:>9.4f}" for k in KEYS)
            print(row)
            for k in KEYS:
                if g: totals[k].append(m[k])
        print(f"  {'─'*71}")
        # Overall mean
        overall = {k: np.mean(totals[k]) if totals[k] else 0.0 for k in KEYS}
        print(f"  {'Tổng/Mean':<13}{len(per_sample):>4}  " + "".join(
            f"{overall[k]:>9.2f}" if k=="hd95" else f"{overall[k]:>9.4f}" for k in KEYS))
        print(bar)

# ═══════════════════════════════════════════════════════════════════════
# 5. VISUALIZATION THEO NHÓM
# ═══════════════════════════════════════════════════════════════════════
def show_bin_analysis(per_sample):
    groups = _bin_groups(per_sample, key="gcam_dice")

    print(f"\n{'─'*60}")
    print("  PHÂN NHÓM THEO DICE GRADCAM (bước đầu)")
    print(f"{'─'*60}")
    for bname, _, _, _ in BINS:
        g  = groups[bname]
        md = np.mean([s["steps"][0]["dice"] for s in g]) if g else 0.0
        print(f"  [{bname}] : {len(g):3d} mẫu  |  mean GradCAM Dice = {md:.4f}")
    print(f"{'─'*60}")

    # n_cols: [Zero Pred] [GradCAM] [Rescue Box] [steps[0]..steps[N-1]] [GT]
    n_cols = 3 + NUM_IPR_STEPS + 1   # 3 cố định (Zero/GradCAM/Rescue) + N steps + GT

    for bname, lo, hi, color in BINS:
        g = groups[bname]
        if not g: continue
        g_sorted = sorted(g, key=lambda s: s["steps"][0]["dice"])
        idxs   = np.linspace(0, len(g_sorted)-1, min(NUM_VIS_PER_BIN, len(g_sorted)), dtype=int)
        chosen = [g_sorted[k] for k in idxs]
        n_rows = len(chosen)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3.2))
        if n_rows == 1: axes = axes[np.newaxis, :]
        fig.suptitle(f"Cụm Dice {bname}  (tổng {len(g)} mẫu — {len(chosen)} minh họa)",
                     fontsize=11, fontweight='bold')

        for ri, s in enumerate(chosen):
            imgd = s["img"] * 0.5 + 0.5

            # Col 0 – Zero Prompt → Mask rỗng (Achievement 1)
            axes[ri,0].imshow(imgd, cmap='gray')
            pred0_ov = np.zeros((*imgd.shape[:2], 4))
            pred0_ov[s["prob0"] > 0.5] = [0.1,0.9,0.1,0.5]
            axes[ri,0].imshow(pred0_ov)
            axes[ri,0].text(imgd.shape[1]*0.5, imgd.shape[0]*0.5, "∅",
                fontsize=36, color="red", fontweight="bold", ha="center", va="center",
                bbox=dict(facecolor="white", alpha=0.6, edgecolor="red", linewidth=2))
            axes[ri,0].set_title("Mask rỗng\n(Pred = ∅)", fontsize=8, fontweight='bold', color='red')
            axes[ri,0].axis('off')

            # Col 1 – GradCAM
            axes[ri,1].imshow(imgd, cmap='gray')
            axes[ri,1].imshow(s["sal"], cmap='jet', alpha=0.4)
            axes[ri,1].set_title("GradCAM", fontsize=8, fontweight='bold')
            axes[ri,1].axis('off')

            # Col 2 – Rescue Box (prompt từ GradCAM)
            pm0 = s["steps"][0]["pm"]
            axes[ri,2].imshow(imgd, cmap='gray')
            axes[ri,2].imshow(np.ma.masked_where(pm0<0.1, pm0), cmap='magma', alpha=0.5)
            axes[ri,2].set_title("Rescue Box", fontsize=8, fontweight='bold')
            axes[ri,2].axis('off')

            # Cols 3..3+N-1 – steps[0]=GradCAM pred, steps[1]=IPR V1, steps[2]=IPR V2
            step_labels = ["GradCAM Pred"] + [f"IPR V{v}" for v in range(1, NUM_IPR_STEPS)]
            for v, step in enumerate(s["steps"]):
                ax = axes[ri, 3+v]
                ax.imshow(imgd, cmap='gray')
                ov = np.zeros((*step["mask"].shape, 4))
                ov[step["mask"]>0.5] = [0.1,0.9,0.1,0.5]
                ax.imshow(ov)
                if step["cx"] is not None:
                    ax.plot(step["cx"], step["cy"], 'o', color='white', ms=4)
                lbl = step_labels[v] if v < len(step_labels) else f"IPR V{v}"
                ax.set_title(f"{lbl}\nD={step['dice']:.3f}", fontsize=8, fontweight='bold')
                ax.axis('off')

            # Col cuối – GT
            ax_gt = axes[ri, n_cols-1]
            ax_gt.imshow(imgd, cmap='gray')
            gt_ov = np.zeros((*s["gt"].shape, 4))
            gt_ov[s["gt"]>0.5] = [0.1,0.4,1.0,0.5]
            ax_gt.imshow(gt_ov)
            ax_gt.set_title("GT", fontsize=8, fontweight='bold')
            ax_gt.axis('off')

        plt.tight_layout()
        fname = f"gradcam_ipr_bin_{bname.replace(' ','').replace('–','_').replace('.','')}.png"
        plt.savefig(fname, dpi=100, bbox_inches='tight')
        plt.show()
        print(f"  ✅ Saved: {fname}")

# ═══════════════════════════════════════════════════════════════════════
# 6. LOAD CHECKPOINT & CHẠY
# ═══════════════════════════════════════════════════════════════════════
model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=USE_ENCODER_PROMPT).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
print(f"✅ Đã load checkpoint: {MODEL_PATH}")

# prompt_mode='zoom_out' để thống nhất với kịch bản test chính (cell 9)
test_ds    = BTXRD_Dataset(TEST_IMAGE_DIR, TEST_JSON_DIR,
                           img_size=IMG_SIZE, is_train=False, prompt_mode='zoom_out')
per_sample = run_full_evaluation(model, test_ds)
print_per_bin_tables(per_sample)
show_bin_analysis(per_sample)
